In [0]:
%pip install catboost lightgbm optuna

In [0]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, cross_val_score, cross_validate, cross_val_predict
from sklearn.metrics import roc_auc_score

from sklearn.preprocessing import OneHotEncoder, TargetEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier, Pool
from sklearn.linear_model import LogisticRegression

import optuna

In [0]:
TARGET = "Will_Buy_EV"
N_SPLITS = 5

cat_columns = [
    "Gender",
    "City_Type",
    "Current_Car_Type"
    # "Home_Charging_Possible",
    # "Subsidy_Available",
    # "Range_Anxiety_Level",
]
num_columns = [
    "Age",
    "Annual_Income_USD",
    "Daily_Commute_km",
    "Number_of_Cars_Owned",
    "Charging_Stations_Near_Home",
    "Charging_Stations_Near_Work",
    "Environmental_Concern_Level",
]

yes_no_columns = ["Home_Charging_Possible", "Subsidy_Available"]
ordinal_columns = ["Range_Anxiety_Level"]

In [0]:
def load_data():
    df_train = pd.read_csv("train.csv")
    df_test = pd.read_csv("test.csv")

    df_test["source"] = "test"
    df_train["source"] = "train"

    df = pd.concat([df_train, df_test], ignore_index=True)

    df[TARGET] = df[TARGET].str.lower().map({"no": 0, "yes": 1})
    return df

In [0]:
def data_splitter(df):
    training_set = df[df["source"] == "train"]
    submission_set = df[df["source"] == "test"].drop(columns=[TARGET])

    X_train, y_train = training_set.drop(columns=[TARGET, "id", "source"]), training_set[TARGET]

    return X_train, y_train, submission_set

In [0]:
df = load_data()

# EDA

## Info

In [0]:
df.info()

In [0]:
df.describe().T

In [0]:
df.isna().sum()

In [0]:
df.head()

## Adversarial Validation

See if the distributions of train and test sets match

In [0]:
X_adv = df.drop(columns=[TARGET, "source", "id"])
y_adv = df["source"].map({"train": 0, "test": 1})

for c in cat_columns:
    X_adv[c] = X_adv[c].astype("category")

cv_adv = StratifiedKFold(5, shuffle=True, random_state=42)
oof_adv = cross_val_predict(
    LGBMClassifier(n_estimators=300, learning_rate=0.05, random_state=42, verbose=-1),
    X_adv,
    y_adv,
    cv=cv_adv,
    method="predict_proba",
)[:, 1]
print(roc_auc_score(y_adv, oof_adv))

## EDA on Features

In [0]:
# Income to Target
plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x='Annual_Income_USD',
    hue=TARGET,
    multiple='fill',
    kde=True,
    bins=18
)

In [0]:
# Daily Commute to Target
plt.figure(figsize=(8,5))

sns.histplot(
    data=df,
    x='Daily_Commute_km',
    hue=TARGET,
    multiple='fill',
    kde=True,
    bins=18
)

In [0]:
df_corr = df.corr(numeric_only=True)

sns.heatmap(
    data=df_corr,
    annot=True,
    fmt=".2f",
)

# Feature Engineering

In [0]:
df.head()

In [0]:
def feature_engineering(df):
    for c in df[yes_no_columns]:
        df[c] = df[c].str.lower().map({"no": 0, "yes": 1})


    df["Range_Anxiety_Level"] = (
        df["Range_Anxiety_Level"].str.lower().map({"low": 0, "medium": 1, "high": 2})
    )

    df['Total_Charging_Stations'] = df['Charging_Stations_Near_Home'] + df['Charging_Stations_Near_Work'] + df['Home_Charging_Possible']

    df['Charging_Stations_Per_Km'] = np.where(
        df['Total_Charging_Stations'] == 0,
        0,
        df['Daily_Commute_km'] / df['Total_Charging_Stations']
    )

    df['Realistic_Concern'] = df['Environmental_Concern_Level'] - df['Range_Anxiety_Level']

    df['Anxiety_Commute_Ratio'] = np.where(
        df['Daily_Commute_km'] == 0,
        0,
        df['Range_Anxiety_Level'] / df['Daily_Commute_km']
    )
    return df

# Baseline model (Logistic Regression)

In [0]:
df.head()

In [0]:
X, y, submission_set = data_splitter(df)

In [0]:
log_reg_preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), cat_columns),
        ('scaler', StandardScaler(), num_columns),
    ],
    remainder="drop"
)

log_reg_pipeline = Pipeline([
    ('preprocessor', log_reg_preprocessor),
    ('logreg', LogisticRegression(max_iter=1000))
])

In [0]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    random_state=42,
    shuffle=True
)

In [0]:
oof = np.zeros(len(X))
submission_results = np.zeros(len(submission_set))
folds_train_results, folds_validation_results = [], []

for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
    X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    log_reg_pipeline.fit(X_train, y_train)
    oof_pred = log_reg_pipeline.predict_proba(X_val)[:,1]
    train_pred = log_reg_pipeline.predict_proba(X_train)[:,1]
    
    oof_auc = roc_auc_score(y_val, oof_pred)
    train_auc = roc_auc_score(y_train, train_pred)

    oof[val_idx] = oof_pred
    folds_train_results.append(train_auc)
    folds_validation_results.append(oof_auc)
    submission_results += log_reg_pipeline.predict_proba(submission_set)[:,1] / N_SPLITS

    print(f"=======FOLD: {fold} =======")
    print(f"TRAIN AUC: {train_auc} | TEST AUC: {oof_auc} | DELTA {oof_auc - train_auc}")

tr, va = np.array(folds_train_results), np.array(folds_validation_results)
print("\n====== CROSS VALIDATION ENDED ======")
print(f"TRAIN AVG AUC: {tr.mean():.5f} +/- {tr.std():.5f} | VAL AVG AUC: {va.mean():.5f} +/- {va.std():.5f}")
print(f"\nOOF AUC: {roc_auc_score(y, oof):.5f}")

## Baseline ROC AUC

TRAIN AVG AUC: 0.93811 +/- 0.00020
VAL AVG AUC: 0.93810 +/- 0.00081

OOF AUC: 0.93809

# XGBoost

In [0]:
df = load_data()
df_ = feature_engineering(df)
X, y, submission_set = data_splitter(df_)

xgb_params = {
    "n_estimators": 1095,
    "max_depth": 8,
    "learning_rate": 0.06757009447016467,
    "subsample": 0.8526064253616809,
    "colsample_bytree": 0.713964646305348,
    "min_child_weight": 39,
    "reg_alpha": 7.741721190554106e-05,
    "reg_lambda": 0.009405232270558986,
    "gamma": 0.6884035558163024,
}

xgb_model = XGBClassifier(**xgb_params, random_state=42, n_jobs=1)

xgb_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), cat_columns),
    ],
    remainder="passthrough",
)

xgb_pipeline = Pipeline([("preprocessor", xgb_preprocessor), ("xgb", xgb_model)])

In [0]:
cv = StratifiedKFold(
    n_splits=N_SPLITS,
    random_state=42,
    shuffle=True
)

oof = np.zeros(len(X))
submission_results = np.zeros(len(submission_set))
folds_train_results, folds_validation_results = [], []

for fold, (tr_idx, val_idx) in enumerate(cv.split(X, y)):
    X_train, y_train = X.iloc[tr_idx], y.iloc[tr_idx]
    X_val, y_val = X.iloc[val_idx], y.iloc[val_idx]

    xgb_pipeline.fit(X_train, y_train)
    oof_pred = xgb_pipeline.predict_proba(X_val)[:,1]
    train_pred = xgb_pipeline.predict_proba(X_train)[:,1]
    
    oof_auc = roc_auc_score(y_val, oof_pred)
    train_auc = roc_auc_score(y_train, train_pred)

    oof[val_idx] = oof_pred
    folds_train_results.append(train_auc)
    folds_validation_results.append(oof_auc)
    submission_results += xgb_pipeline.predict_proba(submission_set)[:,1] / N_SPLITS

    print(f"=======FOLD: {fold} =======")
    print(f"TRAIN AUC: {train_auc} | TEST AUC: {oof_auc} | DELTA {oof_auc - train_auc}")

tr, va = np.array(folds_train_results), np.array(folds_validation_results)
print("\n====== CROSS VALIDATION ENDED ======")
print(f"TRAIN AVG AUC: {tr.mean():.5f} +/- {tr.std():.5f} | VAL AVG AUC: {va.mean():.5f} +/- {va.std():.5f}")
print(f"\nOOF AUC: {roc_auc_score(y, oof):.5f}")

# Submission

In [0]:
submit_df = submission_set.reset_index().rename(columns={"index": "id"}).copy()
submit_df[TARGET] = submission_results

In [0]:
submit_df[['id', TARGET]].to_csv('log_reg_submission_v1.csv', index=False)